# Train ViT5 chuẩn: Phase 1 full fine-tuning → Phase 2 LoRA

Notebook này **giữ nguyên** `data/phase_1` và `data/phase_2`. Phase 1 tạo một full checkpoint cố định; Phase 2 chỉ lưu LoRA adapter, không merge và không ghi đè Phase 1.

Quy trình an toàn:

1. Chạy `phase1_smoke`, sau đó đổi sang `phase1_full`.
2. Lưu/version thư mục `outputs_phase_1/vit5_base/best`.
3. Ở phiên GPU mới, mount checkpoint đó rồi chạy `phase2_smoke`, sau đó `phase2_full`.
4. Chỉ dùng validation để chọn checkpoint. Test được chạy sau khi cấu hình đã khóa.

> Mặc định là smoke test 20 bước. Notebook từ chối ghi vào output không rỗng nếu không resume, để không trộn artifact giữa các lần chạy.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
REPO_URL = "https://github.com/dungcony/text-sumarization.git"
REPO_REF = ""  # Nên điền commit/tag chứa pipeline này trước khi chạy Kaggle.
PROJECT_SUBDIR = Path("tuan 5-6/sumarization")

def locate_project() -> Path:
    cwd = Path.cwd().resolve()
    clone_root = Path("/kaggle/working/text-sumarization")
    candidates = [cwd, *cwd.parents, clone_root / PROJECT_SUBDIR]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate

    if Path("/kaggle/working").is_dir():
        if clone_root.exists():
            raise RuntimeError(f"{clone_root} đã tồn tại nhưng thiếu project con {PROJECT_SUBDIR}.")
        subprocess.run(["git", "clone", REPO_URL, str(clone_root)], check=True)
        if REPO_REF.strip():
            subprocess.run(["git", "-C", str(clone_root), "checkout", REPO_REF.strip()], check=True)
        project = clone_root / PROJECT_SUBDIR
        if not (project / "src").is_dir():
            raise FileNotFoundError(f"Repo đã clone nhưng không có project tại {project}.")
        return project

    raise FileNotFoundError("Không tìm thấy project. Hãy mở notebook từ repo hoặc sửa REPO_URL.")

PROJECT_DIR = locate_project()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)],
    check=True,
)
print(f"Project: {PROJECT_DIR}")
try:
    commit = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    print(f"Git commit: {commit}")
    if Path("/kaggle/working").is_dir() and not REPO_REF.strip():
        print("⚠️ Nên đặt REPO_REF bằng commit này để lần chạy Kaggle có thể tái lập.")
except (OSError, subprocess.CalledProcessError):
    print("Không đọc được Git commit của source tree hiện tại.")


## 1. Bảng điều khiển

Mỗi lần chỉ chọn **một** `MODE`. Khi chạy lại full train, đặt `RUN_TAG` mới thay vì ghi đè output cũ. Với Phase 2 trên Kaggle, nên điền rõ `MANUAL_PHASE1_BEST` để chắc chắn adapter dùng đúng base.


In [ ]:
MODE = "phase1_smoke"
# MODE hợp lệ: phase1_smoke, phase1_full, phase2_smoke, phase2_full, evaluate

RUN_TAG = ""  # Ví dụ: v2 hoặc seed42; để trống cho đường dẫn chuẩn trong guide.
MANUAL_DATA_ROOT = ""  # Thư mục chứa phase_1/ và phase_2/; cần khi data mount từ Kaggle Input.
MANUAL_PHASE1_BEST = ""  # Full checkpoint Phase 1, thư mục best/.
MANUAL_PHASE2_ADAPTER = ""  # LoRA adapter Phase 2, thư mục best/.
RESUME_CHECKPOINT = ""  # Chỉ dùng checkpoint-N của chính run đang tiếp tục.

RUN_TEST_AFTER_FULL_TRAIN = True
RUN_CROSS_PHASE_MATRIX = True
RUN_PHASE1_PRETRAINED_BASELINE = True
REQUIRE_PHASE1_VALIDATION_IMPROVEMENT = True
REQUIRE_PHASE2_VALIDATION_IMPROVEMENT = True
MAX_PHASE1_VALIDATION_ROUGEL_DROP = 1.0  # Đặt None nếu route Phase 1 luôn tắt adapter.
ALLOW_CPU = False  # Full ViT5-base trên CPU rất chậm; chỉ bật khi bạn thực sự chủ động.
EVAL_MAX_SAMPLES = None  # None = toàn split; đặt số nhỏ chỉ để debug, không báo cáo final.

ALLOWED_MODES = {
    "phase1_smoke", "phase1_full",
    "phase2_smoke", "phase2_full", "evaluate",
}
if MODE not in ALLOWED_MODES:
    raise ValueError(f"MODE={MODE!r} không hợp lệ; chọn một trong {sorted(ALLOWED_MODES)}")
if MODE.endswith("_full") and EVAL_MAX_SAMPLES is not None:
    print("⚠️ EVAL_MAX_SAMPLES đang giới hạn evaluation; không dùng kết quả này làm final.")
print(f"MODE={MODE} | RUN_TAG={RUN_TAG or '(mặc định)'}")


## 2. Kiểm tra dữ liệu, phần cứng và artifact

Cell này xác nhận notebook vẫn đọc đúng hai split hiện có. Nó không lọc, trộn hoặc chia lại dữ liệu.


In [ ]:
import gc
import json

import torch

from src.config import apply_overrides, config_to_dict, load_config
from src.data import load_dataset_from_files
from src.evaluator import evaluate_checkpoint
from src.model import verify_adapter_base_dependency
from src.trainer import train
from src.utils import save_json

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else PROJECT_DIR
suffix = f"_{RUN_TAG.strip()}" if RUN_TAG.strip() else ""
PHASE1_OUTPUT = WORK_ROOT / f"outputs_phase_1/vit5_base{suffix}"
PHASE2_OUTPUT = WORK_ROOT / f"outputs_phase_2_lora/vit5_base{suffix}"
SMOKE_ROOT = WORK_ROOT / f"smoke_runs{suffix}"
EVALUATION_ROOT = WORK_ROOT / f"evaluations_two_phase{suffix}"

def is_data_root(path: Path) -> bool:
    required = (
        ("phase_1", "train_*.*"), ("phase_1", "validation_*.*"), ("phase_1", "test_*.*"),
        ("phase_2", "train_*.*"), ("phase_2", "validation_*.*"), ("phase_2", "test_*.*"),
    )
    return path.is_dir() and all(any((path / phase).glob(pattern)) for phase, pattern in required)

def resolve_data_root() -> Path:
    if MANUAL_DATA_ROOT.strip():
        supplied = Path(MANUAL_DATA_ROOT).expanduser().resolve()
        for candidate in (supplied, supplied / "data"):
            if is_data_root(candidate):
                return candidate
        raise FileNotFoundError(
            f"MANUAL_DATA_ROOT không chứa đủ phase_1/phase_2 splits: {supplied}"
        )

    local_data = (PROJECT_DIR / "data").resolve()
    if is_data_root(local_data):
        return local_data

    kaggle_input = Path("/kaggle/input")
    matches = set()
    if kaggle_input.is_dir():
        for phase1_dir in kaggle_input.rglob("phase_1"):
            candidate = phase1_dir.parent.resolve()
            if phase1_dir.is_dir() and is_data_root(candidate):
                matches.add(candidate)
    if len(matches) == 1:
        return matches.pop()
    if not matches:
        raise FileNotFoundError(
            "Không tìm thấy data/phase_1 và data/phase_2. Hãy Add Input rồi đặt MANUAL_DATA_ROOT."
        )
    choices = "\n".join(f"  - {path}" for path in sorted(matches))
    raise RuntimeError(f"Tìm thấy nhiều data root; hãy chọn MANUAL_DATA_ROOT:\n{choices}")

DATA_ROOT = resolve_data_root()

def config_with_data_root(cfg):
    phase_dir = "phase_1" if cfg.phase.name == "phase_1" else "phase_2"
    split_root = DATA_ROOT / phase_dir
    return apply_overrides(cfg, {
        "data.train_file": str(split_root / "train_*.*"),
        "data.valid_file": str(split_root / "validation_*.*"),
        "data.test_file": str(split_root / "test_*.*"),
    })

def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_split_counts(config_path: str) -> dict[str, int]:
    cfg = config_with_data_root(load_config(config_path))
    raw = load_dataset_from_files(
        train_file=cfg.data.train_file,
        valid_file=cfg.data.valid_file,
        test_file=cfg.data.test_file,
    )
    counts = {name: len(values) for name, values in raw.items()}
    del raw
    return counts

expected = {
    "phase_1": {"train": 10775, "validation": 1348, "test": 1344},
    "phase_2": {"train": 6909, "validation": 859, "test": 871},
}
actual = {
    "phase_1": load_split_counts("configs/vit5_base_phase_1.yaml"),
    "phase_2": load_split_counts("configs/vit5_base_phase_2_lora.yaml"),
}
if actual != expected:
    raise RuntimeError(f"Số dòng dữ liệu đã thay đổi. Expected={expected}; actual={actual}")

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("Không tìm thấy CUDA GPU. Hãy bật GPU accelerator hoặc đặt ALLOW_CPU=True.")
print(json.dumps(actual, ensure_ascii=False, indent=2))
print(f"Data root: {DATA_ROOT}")
print(f"Thiết bị: {device}")
print(f"Phase 1 output: {PHASE1_OUTPUT}")
print(f"Phase 2 output: {PHASE2_OUTPUT}")


In [ ]:
FULL_WEIGHT_FILES = (
    "model.safetensors", "model.safetensors.index.json",
    "pytorch_model.bin", "pytorch_model.bin.index.json",
)
TOKENIZER_FILES = ("spiece.model", "tokenizer.json")
ADAPTER_WEIGHT_FILES = ("adapter_model.safetensors", "adapter_model.bin")

def is_full_checkpoint(path: Path) -> bool:
    return path.is_dir() and (path / "config.json").is_file() and any(
        (path / name).is_file() for name in FULL_WEIGHT_FILES
    ) and any((path / name).is_file() for name in TOKENIZER_FILES) and not (
        path / "adapter_config.json"
    ).exists()

def is_adapter_checkpoint(path: Path) -> bool:
    return path.is_dir() and (path / "adapter_config.json").is_file() and any(
        (path / name).is_file() for name in ADAPTER_WEIGHT_FILES
    )

def resolve_artifact(manual: str, preferred: Path, kind: str) -> Path:
    validator = is_full_checkpoint if kind == "full" else is_adapter_checkpoint
    marker = "config.json" if kind == "full" else "adapter_config.json"

    if manual.strip():
        path = Path(manual).expanduser().resolve()
        if not validator(path):
            raise FileNotFoundError(f"Artifact {kind} không hợp lệ: {path}")
        return path

    preferred = preferred.resolve()
    if validator(preferred):
        return preferred

    search_root = Path("/kaggle/input")
    matches = []
    if search_root.is_dir():
        matches = sorted({p.parent.resolve() for p in search_root.rglob(marker) if validator(p.parent)})
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f"Không tìm thấy artifact {kind}. Hãy điền đường dẫn MANUAL tương ứng."
        )
    choices = "\n".join(f"  - {path}" for path in matches[:20])
    raise RuntimeError(f"Tìm thấy nhiều artifact {kind}; hãy chọn MANUAL:\n{choices}")

def guard_training_output(output_dir: Path, resume_checkpoint: str) -> None:
    resume = resume_checkpoint.strip()
    if resume:
        checkpoint = Path(resume).expanduser().resolve()
        if not (checkpoint / "trainer_state.json").is_file():
            raise FileNotFoundError(f"Resume checkpoint không hợp lệ: {checkpoint}")
        if output_dir.resolve() not in checkpoint.parents:
            raise ValueError("RESUME_CHECKPOINT phải nằm trong đúng output_dir của run hiện tại.")
        return
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(
            f"Output đã có dữ liệu: {output_dir}. Hãy đặt RUN_TAG mới hoặc dùng RESUME_CHECKPOINT."
        )

phase1_best = None
phase2_adapter = None
if MODE in {"phase2_smoke", "phase2_full", "evaluate"}:
    phase1_best = resolve_artifact(
        MANUAL_PHASE1_BEST, PHASE1_OUTPUT / "best", "full"
    )
    print(f"Phase 1 base: {phase1_best}")
if MODE == "evaluate":
    phase2_adapter = resolve_artifact(
        MANUAL_PHASE2_ADAPTER, PHASE2_OUTPUT / "best", "adapter"
    )
    if not verify_adapter_base_dependency(phase1_best, phase2_adapter):
        raise RuntimeError("Notebook chuẩn yêu cầu adapter manifest có base fingerprint.")
    print(f"Phase 2 adapter: {phase2_adapter}")


## 3. Huấn luyện

Smoke mode dùng 64 mẫu train, 32 mẫu validation và 20 optimizer steps. Full mode dùng toàn bộ dữ liệu/config YAML. Phase 2 sẽ fail-fast nếu bất kỳ tham số trainable nào không phải `lora_*`.


In [ ]:
def build_training_config(
    config_path: str,
    output_dir: Path,
    *,
    base_model: Path | None = None,
    smoke: bool = False,
):
    cfg = config_with_data_root(load_config(config_path))
    overrides = {"training.output_dir": str(output_dir)}
    if base_model is not None:
        overrides["model.name_or_path"] = str(base_model)
    if RESUME_CHECKPOINT.strip():
        overrides["training.resume_from_checkpoint"] = RESUME_CHECKPOINT.strip()
    if smoke:
        overrides.update({
            "training.max_steps": 20,
            "training.eval_strategy": "steps",
            "training.eval_steps": 10,
            "training.save_strategy": "steps",
            "training.save_steps": 10,
            "training.logging_steps": 1,
            "data.max_train_samples": 64,
            "data.max_eval_samples": 32,
        })
    return apply_overrides(cfg, overrides)

train_metrics = None
active_config = None

if MODE in {"phase1_smoke", "phase1_full"}:
    smoke = MODE == "phase1_smoke"
    output_dir = SMOKE_ROOT / "phase1" if smoke else PHASE1_OUTPUT
    guard_training_output(output_dir, RESUME_CHECKPOINT)
    active_config = build_training_config(
        "configs/vit5_base_phase_1.yaml", output_dir, smoke=smoke
    )
    print(json.dumps(config_to_dict(active_config), ensure_ascii=False, indent=2))
    train_metrics = train(active_config)
    phase1_best = output_dir / "best"
    if not is_full_checkpoint(phase1_best):
        raise RuntimeError(f"Không tạo được full checkpoint Phase 1: {phase1_best}")
    print(f"✅ Phase 1 full checkpoint: {phase1_best}")
    clear_memory()

elif MODE in {"phase2_smoke", "phase2_full"}:
    smoke = MODE == "phase2_smoke"
    output_dir = SMOKE_ROOT / "phase2_lora" if smoke else PHASE2_OUTPUT
    guard_training_output(output_dir, RESUME_CHECKPOINT)
    active_config = build_training_config(
        "configs/vit5_base_phase_2_lora.yaml",
        output_dir,
        base_model=phase1_best,
        smoke=smoke,
    )
    if not active_config.lora.enabled or active_config.training.freeze_encoder:
        raise RuntimeError("Phase 2 chuẩn phải bật LoRA và không dùng freeze_encoder.")
    print(json.dumps(config_to_dict(active_config), ensure_ascii=False, indent=2))
    train_metrics = train(active_config)
    phase2_adapter = output_dir / "best"
    if not is_adapter_checkpoint(phase2_adapter):
        raise RuntimeError(f"Không tạo được LoRA adapter Phase 2: {phase2_adapter}")
    if not verify_adapter_base_dependency(phase1_best, phase2_adapter):
        raise RuntimeError("Adapter vừa train thiếu base fingerprint.")
    if not is_full_checkpoint(phase1_best):
        raise RuntimeError("Checkpoint Phase 1 không còn nguyên vẹn sau Phase 2.")
    print(f"✅ Phase 2 LoRA adapter: {phase2_adapter}")
    print(f"✅ Phase 1 vẫn là full checkpoint riêng: {phase1_best}")
    clear_memory()

else:
    print("MODE=evaluate: bỏ qua train.")

if train_metrics is not None:
    print(json.dumps(train_metrics, ensure_ascii=False, indent=2))


## 4. Reload và đánh giá đúng split

- Smoke mode chỉ đánh giá lại **validation** để kiểm tra artifact có reload được.
- `phase1_full` chỉ chấm Phase 1 test sau khi train xong.
- `phase2_full`/`evaluate` so sánh Phase 1 và Phase 1 + LoRA trên **cùng từng test split, cùng generation config**.
- Mỗi mẫu được xuất JSONL với toàn bộ article/reference/prediction để audit lỗi factuality.


In [ ]:
def evaluation_config(config_path: str, base_model: Path):
    cfg = config_with_data_root(load_config(config_path))
    overrides = {
        "model.name_or_path": str(base_model),
        "training.per_device_eval_batch_size": 1,
    }
    if EVAL_MAX_SAMPLES is not None:
        overrides["data.max_eval_samples"] = EVAL_MAX_SAMPLES
    return apply_overrides(cfg, overrides)

def run_evaluation(
    label: str,
    model_path: Path,
    cfg,
    split: str,
    *,
    base_model: Path | None = None,
) -> dict[str, float]:
    print(f"\n===== {label} =====")
    metrics = evaluate_checkpoint(
        model_path=model_path,
        config=cfg,
        output_dir=EVALUATION_ROOT / label,
        export_predictions=True,
        split=split,
        base_model_path=base_model,
    )
    clear_memory()
    return metrics

evaluation_matrix = {}

if MODE == "phase1_smoke":
    cfg = evaluation_config("configs/vit5_base_phase_1.yaml", phase1_best)
    cfg = apply_overrides(cfg, {"data.max_eval_samples": 32})
    evaluation_matrix["phase1_smoke_validation"] = run_evaluation(
        "phase1_smoke_validation", phase1_best, cfg, "validation"
    )

elif MODE == "phase2_smoke":
    cfg = evaluation_config("configs/vit5_base_phase_2_lora.yaml", phase1_best)
    cfg = apply_overrides(cfg, {"data.max_eval_samples": 32})
    evaluation_matrix["phase2_lora_smoke_validation"] = run_evaluation(
        "phase2_lora_smoke_validation",
        phase2_adapter,
        cfg,
        "validation",
        base_model=phase1_best,
    )

elif MODE == "phase1_full":
    cfg = evaluation_config("configs/vit5_base_phase_1.yaml", phase1_best)
    if RUN_PHASE1_PRETRAINED_BASELINE:
        evaluation_matrix["pretrained_on_phase1_validation"] = run_evaluation(
            "pretrained_on_phase1_validation", Path("VietAI/vit5-base"), cfg, "validation"
        )
        evaluation_matrix["phase1_on_phase1_validation"] = run_evaluation(
            "phase1_on_phase1_validation", phase1_best, cfg, "validation"
        )
        base_val = evaluation_matrix["pretrained_on_phase1_validation"]["validation_rougeL"]
        phase1_val = evaluation_matrix["phase1_on_phase1_validation"]["validation_rougeL"]
        val_delta = round(phase1_val - base_val, 4)
        print(f"Phase 1 validation ΔROUGE-L (fine-tuned - pretrained): {val_delta:+.4f}")
        if REQUIRE_PHASE1_VALIDATION_IMPROVEMENT and val_delta <= 0:
            save_json(
                {"metrics": evaluation_matrix, "phase1_validation_rougeL_delta": val_delta},
                EVALUATION_ROOT / "phase1_validation_gate_failed.json",
            )
            raise RuntimeError("Phase 1 không cải thiện validation ROUGE-L; dừng trước test.")
    if RUN_TEST_AFTER_FULL_TRAIN:
        evaluation_matrix["phase1_on_phase1_test"] = run_evaluation(
            "phase1_on_phase1_test", phase1_best, cfg, "test"
        )

elif MODE in {"phase2_full", "evaluate"}:
    phase1_cfg = evaluation_config("configs/vit5_base_phase_1.yaml", phase1_best)
    phase2_cfg = evaluation_config("configs/vit5_base_phase_2_lora.yaml", phase1_best)

    evaluation_matrix["phase1_on_phase2_validation"] = run_evaluation(
        "phase1_on_phase2_validation", phase1_best, phase2_cfg, "validation"
    )
    evaluation_matrix["phase1_lora_on_phase2_validation"] = run_evaluation(
        "phase1_lora_on_phase2_validation",
        phase2_adapter,
        phase2_cfg,
        "validation",
        base_model=phase1_best,
    )
    base_val = evaluation_matrix["phase1_on_phase2_validation"]["validation_rougeL"]
    lora_val = evaluation_matrix["phase1_lora_on_phase2_validation"]["validation_rougeL"]
    val_delta = round(lora_val - base_val, 4)
    print(f"Phase 2 validation ΔROUGE-L (LoRA - Phase 1): {val_delta:+.4f}")

    if REQUIRE_PHASE2_VALIDATION_IMPROVEMENT and val_delta <= 0:
        save_json(
            {"metrics": evaluation_matrix, "phase2_validation_rougeL_delta": val_delta},
            EVALUATION_ROOT / "validation_gate_failed.json",
        )
        raise RuntimeError(
            "LoRA không cải thiện Phase 2 validation ROUGE-L; dừng trước khi xem test."
        )

    if RUN_CROSS_PHASE_MATRIX:
        evaluation_matrix["phase1_on_phase1_validation"] = run_evaluation(
            "phase1_on_phase1_validation", phase1_best, phase1_cfg, "validation"
        )
        evaluation_matrix["phase1_lora_on_phase1_validation"] = run_evaluation(
            "phase1_lora_on_phase1_validation",
            phase2_adapter,
            phase1_cfg,
            "validation",
            base_model=phase1_best,
        )
        p1_base_val = evaluation_matrix["phase1_on_phase1_validation"]["validation_rougeL"]
        p1_lora_val = evaluation_matrix["phase1_lora_on_phase1_validation"]["validation_rougeL"]
        retention_drop = round(p1_base_val - p1_lora_val, 4)
        print(f"Phase 1 validation ROUGE-L drop khi bật LoRA: {retention_drop:+.4f}")
        if (
            MAX_PHASE1_VALIDATION_ROUGEL_DROP is not None
            and retention_drop > MAX_PHASE1_VALIDATION_ROUGEL_DROP
        ):
            save_json(
                {"metrics": evaluation_matrix, "phase1_validation_rougeL_drop": retention_drop},
                EVALUATION_ROOT / "retention_validation_gate_failed.json",
            )
            raise RuntimeError(
                "LoRA làm giảm Phase 1 validation quá ngưỡng; dừng trước khi xem test."
            )

    if RUN_TEST_AFTER_FULL_TRAIN:
        evaluation_matrix["phase1_on_phase2_test"] = run_evaluation(
            "phase1_on_phase2_test", phase1_best, phase2_cfg, "test"
        )
        evaluation_matrix["phase1_lora_on_phase2_test"] = run_evaluation(
            "phase1_lora_on_phase2_test",
            phase2_adapter,
            phase2_cfg,
            "test",
            base_model=phase1_best,
        )

    if RUN_TEST_AFTER_FULL_TRAIN and RUN_CROSS_PHASE_MATRIX:
        evaluation_matrix["phase1_on_phase1_test"] = run_evaluation(
            "phase1_on_phase1_test", phase1_best, phase1_cfg, "test"
        )
        evaluation_matrix["phase1_lora_on_phase1_test"] = run_evaluation(
            "phase1_lora_on_phase1_test",
            phase2_adapter,
            phase1_cfg,
            "test",
            base_model=phase1_best,
        )

if evaluation_matrix:
    comparison_deltas = {}
    if {"pretrained_on_phase1_validation", "phase1_on_phase1_validation"} <= evaluation_matrix.keys():
        baseline = evaluation_matrix["pretrained_on_phase1_validation"]
        candidate = evaluation_matrix["phase1_on_phase1_validation"]
        comparison_deltas["phase1_validation_finetuned_minus_pretrained"] = {
            metric: round(candidate[f"validation_{metric}"] - baseline[f"validation_{metric}"], 4)
            for metric in ("rouge1", "rouge2", "rougeL")
        }
    if {"phase1_on_phase2_validation", "phase1_lora_on_phase2_validation"} <= evaluation_matrix.keys():
        baseline = evaluation_matrix["phase1_on_phase2_validation"]
        candidate = evaluation_matrix["phase1_lora_on_phase2_validation"]
        comparison_deltas["phase2_validation_lora_minus_phase1"] = {
            metric: round(candidate[f"validation_{metric}"] - baseline[f"validation_{metric}"], 4)
            for metric in ("rouge1", "rouge2", "rougeL")
        }
    if {"phase1_on_phase1_validation", "phase1_lora_on_phase1_validation"} <= evaluation_matrix.keys():
        baseline = evaluation_matrix["phase1_on_phase1_validation"]
        candidate = evaluation_matrix["phase1_lora_on_phase1_validation"]
        comparison_deltas["phase1_validation_retention_lora_minus_base"] = {
            metric: round(candidate[f"validation_{metric}"] - baseline[f"validation_{metric}"], 4)
            for metric in ("rouge1", "rouge2", "rougeL")
        }
    if {"phase1_on_phase2_test", "phase1_lora_on_phase2_test"} <= evaluation_matrix.keys():
        baseline = evaluation_matrix["phase1_on_phase2_test"]
        candidate = evaluation_matrix["phase1_lora_on_phase2_test"]
        comparison_deltas["phase2_lora_minus_phase1"] = {
            metric: round(candidate[f"test_{metric}"] - baseline[f"test_{metric}"], 4)
            for metric in ("rouge1", "rouge2", "rougeL")
        }
    if {"phase1_on_phase1_test", "phase1_lora_on_phase1_test"} <= evaluation_matrix.keys():
        baseline = evaluation_matrix["phase1_on_phase1_test"]
        candidate = evaluation_matrix["phase1_lora_on_phase1_test"]
        comparison_deltas["phase1_retention_adapter_minus_base"] = {
            metric: round(candidate[f"test_{metric}"] - baseline[f"test_{metric}"], 4)
            for metric in ("rouge1", "rouge2", "rougeL")
        }
    save_json(
        {"metrics": evaluation_matrix, "deltas": comparison_deltas},
        EVALUATION_ROOT / "cross_phase_results.json",
    )
    rows = []
    for label, metrics in evaluation_matrix.items():
        prefix = "validation" if "validation" in label else "test"
        rows.append({
            "run": label,
            "rouge1": metrics.get(f"{prefix}_rouge1"),
            "rouge2": metrics.get(f"{prefix}_rouge2"),
            "rougeL": metrics.get(f"{prefix}_rougeL"),
            "loss": metrics.get(f"{prefix}_loss"),
        })
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except (ImportError, NameError):
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if comparison_deltas:
        print("\nDelta candidate - baseline (điểm dương tốt hơn):")
        print(json.dumps(comparison_deltas, ensure_ascii=False, indent=2))
    print(f"Đã lưu metrics và predictions tại: {EVALUATION_ROOT}")
else:
    print("Không chạy evaluation trong MODE/cấu hình hiện tại.")


## 5. Cách đọc kết quả

Điều kiện tối thiểu trước khi chấp nhận adapter:

- `phase1_lora_on_phase2_test` phải tốt hơn `phase1_on_phase2_test` khi dùng cùng scorer/cấu hình sinh.
- Thư mục Phase 1 vẫn là full checkpoint; Phase 2 chỉ có adapter và `adapter_manifest.json`.
- Đọc file `predictions_test.jsonl` để kiểm tra sai entity, con số, phủ định, thuốc/liều và hallucination. ROUGE không đủ để kết luận an toàn y tế.
- Điểm cũ trước bản sửa tokenizer Unicode là **LEGACY**, không so trực tiếp với điểm mới.

Inference Phase 2 sau khi train:

```python
from src.predict import summarize

summary = summarize(
    text=article,
    base_model_path=phase1_best,
    adapter_path=phase2_adapter,
    config=load_config("configs/vit5_base_phase_2_lora.yaml"),
)
```
